# SASRec - Self-Attentive Sequential Recommender on MovieLens

Модель последовательностей с нуля: статичный эмбеддинг юзера заменён на **causal self-attention** над его историей лайков - предсказываем следующий фильм. «Токены» = фильмы (как GPT, но для рекомендаций); трансформер играет роль **динамической** user-башни.

## 0. Импорты и загрузка данных

In [1]:
import pandas as pd
import random
import torch.nn as nn
import torch
from torch.optim import Adam
import torch.nn.functional as F
import matplotlib.pyplot as plt
import math
import copy
%matplotlib inline

In [2]:
import kagglehub
dataset_path = kagglehub.dataset_download('abhikjha/movielens-100k') + '/ml-latest-small'
print(dataset_path)

D:\kaggle_cache\datasets\abhikjha\movielens-100k\versions\1/ml-latest-small


## 1. Данные и разведка

MovieLens (ml-latest-small). Берём позитивы (оценки ≥ 4.0) и работаем с **последовательностью лайков** каждого юзера, упорядоченной по времени.

In [3]:
movies = pd.read_csv(dataset_path + '/movies.csv')

In [4]:
ratings = pd.read_csv(dataset_path + '/ratings.csv')

In [5]:
ratings['userId'].value_counts()

userId
414    2698
599    2478
474    2108
448    1864
274    1346
       ... 
442      20
278      20
147      20
320      20
53       20
Name: count, Length: 610, dtype: int64

In [6]:
ratings['movieId'].value_counts()

movieId
356       329
318       317
296       307
593       279
2571      278
         ... 
160341      1
160527      1
160836      1
163937      1
135534      1
Name: count, Length: 9724, dtype: int64

In [7]:
user_count = len(ratings['userId'].drop_duplicates())

In [8]:
movie_count = len(ratings['movieId'].drop_duplicates())

In [9]:
ratings['rating'].value_counts()

rating
4.0    26818
3.0    20047
5.0    13211
3.5    13136
4.5     8551
2.0     7551
2.5     5550
1.0     2811
1.5     1791
0.5     1370
Name: count, dtype: int64

In [10]:
len(ratings)

100836

In [11]:
potential_ratings = user_count * movie_count
(len(ratings) / potential_ratings) * 100

1.6999683055613624

In [12]:
len(ratings[ratings['rating']>= 4.0])

48580

In [13]:
genres = movies['genres'].str.split('|').explode().unique()

In [14]:
movies_indexed = movies.set_index('movieId')
genres_df = movies_indexed['genres'].str.get_dummies('|').drop(columns=['(no genres listed)'], errors='ignore')
genres_df = genres_df.reset_index()

## 2. Постановка задачи

**Next-item prediction:** по префиксу `[фильм₁…фильмₜ₋₁]` предсказать `фильмₜ`. Как языковая модель, только токены - фильмы. Учим предсказывать следующий на **каждой** позиции (сдвиг таргета на 1).

In [15]:
pos_ratings = ratings[ratings['rating']>= 4.0]

In [16]:
result = pd.merge(ratings, movies, on='movieId').drop(columns=['genres'])

In [17]:
pos_result = pd.merge(pos_ratings, movies, on='movieId').drop(columns=['genres'])

## 3. Перенумерация ID

userId → 0..U−1. movieId → **1..M**, индекс **0 зарезервирован под паддинг** (`padding_idx=0`). Сдвиг на +1 нужен, чтобы отличать «пусто» от реального фильма.

In [18]:
userIds = result['userId'].drop_duplicates()
user2idx = {id: i for i, id in enumerate(userIds)}
movieIds = result['movieId'].drop_duplicates()
movie2idx = {id: i for i, id in enumerate(movieIds, start=1)}

In [19]:
result['userIdx'] = result['userId'].map(user2idx)
result['movieIdx'] = result['movieId'].map(movie2idx)

In [20]:
pos_result['userIdx'] = pos_result['userId'].map(user2idx)
pos_result['movieIdx'] = pos_result['movieId'].map(movie2idx)

In [21]:
genres_df['movieIdx'] = genres_df['movieId'].map(movie2idx)
genres_df = genres_df.dropna(subset=['movieIdx']).sort_values('movieIdx')
genre_cols = [c for c in genres_df.columns if c not in ('movieId', 'movieIdx')]
genres_tensor = torch.tensor(genres_df[genre_cols].values, dtype=torch.float32)

## 4. Split по времени (leave-two-out) и последовательности

Внутри каждого юзера по времени: **последний** лайк → test, **предпоследний** → val, остальное → train. Из train-лайков строим последовательность, обрезаем до `max_len`, **right-pad** нулями. По времени, а не случайно - чтобы не было утечки будущего.

In [22]:
pos_result = pos_result.sort_values(by=['userIdx', 'timestamp'], ascending=[True, True])

In [23]:
pos_result['row_num'] = pos_result.groupby('userIdx').cumcount()

In [24]:
user_sizes = pos_result.groupby('userIdx').size().reset_index(name='total_count')

In [25]:
pos_result = pd.merge(pos_result, user_sizes, on='userIdx')

In [26]:
pos_result['pct_position'] = pos_result['row_num'] / pos_result['total_count']

In [27]:
test  = pos_result[pos_result['row_num'] == pos_result['total_count'] - 1]
val   = pos_result[pos_result['row_num'] == pos_result['total_count'] - 2]
train = pos_result[pos_result['row_num'] <  pos_result['total_count'] - 2]

In [28]:
columns_to_drop = ['row_num', 'total_count', 'pct_position']
train.drop(columns=columns_to_drop, inplace=True)
val.drop(columns=columns_to_drop, inplace=True)
test.drop(columns=columns_to_drop, inplace=True)

C:\Users\Computer\AppData\Local\Temp\ipykernel_32324\1752980561.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train.drop(columns=columns_to_drop, inplace=True)
C:\Users\Computer\AppData\Local\Temp\ipykernel_32324\1752980561.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  val.drop(columns=columns_to_drop, inplace=True)
C:\Users\Computer\AppData\Local\Temp\ipykernel_32324\1752980561.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-

In [29]:
lengths = train.groupby('userIdx').size()
print(lengths.describe())
print(lengths.quantile([0.5, 0.8, 0.9, 0.95]))

count     608.000000
mean       77.899671
std       109.408962
min         1.000000
25%        19.000000
50%        38.000000
75%        90.500000
max      1225.000000
dtype: float64
0.50     38.0
0.80    114.0
0.90    198.6
0.95    269.3
dtype: float64


In [30]:
max_len = 50
user_seqs = {}
for u, seq in train.groupby('userIdx')['movieIdx'].apply(list).items():
    seq = seq[-max_len:]                      
    pad = [0] * (max_len - len(seq))
    seq = seq + pad                           
    user_seqs[u] = seq

In [31]:
users_order = list(user_seqs.keys())              
seq_tensor  = torch.tensor([user_seqs[u] for u in users_order])

In [32]:
seq_tensor.shape

torch.Size([608, 50])

In [33]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
batch_size = 32
input  = seq_tensor[:, :-1]
target = seq_tensor[:, 1:]

In [34]:
def get_batch():
    # generate a small batch of data of inputs x and targets y
    ix = torch.randint(0, seq_tensor.shape[0], (batch_size,))
    xb = input[ix]
    yb = target[ix]
    x, y = xb.to(device), yb.to(device)
    return x, y

In [35]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    #for split in ['train', 'val']:
    losses = torch.zeros(eval_iters)
    for k in range(eval_iters):
        X, Y = get_batch()
        logits, loss = model(X, Y)
        losses[k] = loss.item()
    out = losses.mean()
    model.train()
    return out

## 5. Модель - SASRec (causal self-attention)

Item- и позиционные эмбеддинги → стек блоков (multi-head **causal** self-attention + FFN, residual + LayerNorm) → `lm_head` в логиты по всем фильмам. Causal mask: позиция видит только прошлое. Выход **последней реальной** позиции = «состояние юзера» для предсказания следующего.

In [36]:
n_emb = 128
n_head = 4
n_layer = 4
dropout = 0.3
eval_iters = 200
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
g = torch.Generator().manual_seed(2147483647)

In [37]:
class Head(nn.Module):
    """One head of self-attention"""

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_emb, head_size, bias=False)
        self.query = nn.Linear(n_emb, head_size, bias=False)
        self.value = nn.Linear(n_emb, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(max_len,max_len)))

        self.dropout = nn.Dropout(dropout)
    

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)
        #Compute attenttion scores ("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) ---> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim = -1)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B, T, C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T,  C)
        return out 


In [38]:
class MultiHeadAttention(nn.Module):
    """multiple heads ofd self-attention in parallel"""
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, n_emb)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

In [39]:
class FeedForward(nn.Module):
    """a simple linear layer followed by a non-linearity"""

    def __init__(self, n_emb):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_emb, 4 * n_emb),
            nn.ReLU(),
            nn.Linear(4 * n_emb, n_emb),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


In [40]:
class Block(nn.Module):
    """Transformer block: communication followed by computation"""

    def __init__(self, n_emb, n_head):
        # n_emd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_emb // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_emb)
        self.ln1 = nn.LayerNorm(n_emb)
        self.ln2 = nn.LayerNorm(n_emb)
    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [41]:
class SASRecModel(nn.Module):

    def __init__(self, user_size, movie_size):
        super().__init__()
        # item (token) + positional embeddings -> transformer blocks -> lm_head over all movies
        #self.user_emb = nn.Embedding(user_size, n_dim) #In future we can try add 
        self.movie_emb = nn.Embedding(movie_size + 1, n_emb, padding_idx=0)  # +1 for padding token
        self.position_embedding_table = nn.Embedding(max_len, n_emb)
        self.blocks = nn.Sequential(*[Block(n_emb, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_emb) #final layer norm
        self.lm_head = nn.Linear(n_emb, movie_size + 1)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.movie_emb(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb #(B, T, C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets, ignore_index=0)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            #crop  idx to the last block_size tokens
            idx_cond = idx[:, -max_len:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = SASRecModel(len(user2idx), (len(movie2idx)))
m = model.to(device)

## 6. Обучение (next-item) + best-by-val

Предсказываем следующий фильм на каждой позиции, loss = `cross_entropy(ignore_index=0)` - паддинг-таргеты игнорируются. Лучший чекпоинт - по **val HR@50**. Метрики: **HR@50 / NDCG@50** (leave-one-out, маска уже виденного).

In [42]:
# optimizer is now created inside the training cell (keeps model->optimizer order correct)


In [43]:
import math, copy

# ---- per-user histories from the leave-one-out splits (train already time-sorted) ----
train_by_user = train.groupby('userIdx')['movieIdx'].apply(list).to_dict()
val_by_user   = val.set_index('userIdx')['movieIdx'].to_dict()    # one held-out item per user
test_by_user  = test.set_index('userIdx')['movieIdx'].to_dict()

@torch.no_grad()
def evaluate(split, k_top=50):
    """Leave-one-out Recall@k & NDCG@k (one held-out item per user).
       split='val' : predict the val item from the user's train sequence.
       split='test': predict the test item from train + val context."""
    model.eval()
    targets = val_by_user if split == 'val' else test_by_user
    sum_recall, sum_ndcg, n_users = 0.0, 0.0, 0
    for u, target in targets.items():
        ctx = list(train_by_user.get(u, []))              # history before the held-out item
        if split == 'test' and u in val_by_user:
            ctx = ctx + [val_by_user[u]]                  # test also conditions on the val item
        if len(ctx) == 0:
            continue                                      # no history -> skip
        seen = set(ctx)                                   # already-seen items (to mask)
        ctx = ctx[-max_len:]                              # keep most recent max_len
        k = len(ctx)
        seq = ctx + [0] * (max_len - k)                   # right-pad with pad token 0
        x = torch.tensor([seq], device=device)            # (1, max_len)
        logits, _ = model(x)                              # (1, max_len, num_movies+1)
        scores = logits[0, k - 1].clone()                 # LAST REAL position (not -1!) predicts next
        scores[list(seen)] = float('-inf')                # never recommend seen items
        scores[0] = float('-inf')                         # never recommend the padding slot
        topk = torch.topk(scores, k_top).indices.tolist()
        if target in topk:
            sum_recall += 1.0
            rank = topk.index(target)                     # 0-based rank in the top-k
            sum_ndcg += 1.0 / math.log2(rank + 2)         # IDCG = 1 for a single relevant item
        n_users += 1
    model.train()
    return sum_recall / n_users, sum_ndcg / n_users


In [ ]:
import copy
# optimizer created HERE (right after the model) so model->optimizer->train order is always correct
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

best_val_recall = -1.0
best_state = None

for iter in range(max_iters):
    # periodic monitoring: train loss + val Recall; keep best checkpoint by val Recall
    if iter % eval_interval == 0:
        train_loss = estimate_loss()
        val_recall, val_ndcg = evaluate('val')                      # toggles eval()/train() inside
        print(f"step {iter}: train loss {train_loss:.4f}  val Recall@50 {val_recall*100:.2f}%  NDCG@50 {val_ndcg*100:.2f}%")
        if val_recall > best_val_recall:
            best_val_recall = val_recall
            best_state = copy.deepcopy(model.state_dict())          # snapshot of the best weights

    xb, yb = get_batch()
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True) 
    loss.backward()
    optimizer.step()

model.load_state_dict(best_state)                                   # restore best-by-val checkpoint
print(f"best val Recall@50: {best_val_recall*100:.2f}%")


step 0: train loss 9.3483  val Recall@50 0.33%  NDCG@50 0.08%
step 500: train loss 5.7191  val Recall@50 15.13%  NDCG@50 4.92%
step 1000: train loss 2.8611  val Recall@50 16.94%  NDCG@50 5.36%
step 1500: train loss 0.8885  val Recall@50 14.14%  NDCG@50 4.97%
step 2000: train loss 0.2347  val Recall@50 13.49%  NDCG@50 4.82%
step 2500: train loss 0.0959  val Recall@50 13.49%  NDCG@50 4.68%
step 3000: train loss 0.0555  val Recall@50 12.01%  NDCG@50 4.29%
step 3500: train loss 0.0455  val Recall@50 12.66%  NDCG@50 4.50%
step 4000: train loss 0.0401  val Recall@50 12.17%  NDCG@50 4.22%
step 4500: train loss 0.0388  val Recall@50 11.51%  NDCG@50 4.13%
best val Recall@50: 16.94%


In [45]:
# ---- final metrics on the held-out TEST item ----
test_recall, test_ndcg = evaluate('test')
val_recall,  val_ndcg  = evaluate('val')
print(f'random Recall@50 baseline ~ {50/len(movie2idx)*100:.2f}%')
print(f'VAL   Recall@50 {val_recall*100:.2f}%   NDCG@50 {val_ndcg*100:.2f}%')
print(f'TEST  Recall@50 {test_recall*100:.2f}%   NDCG@50 {test_ndcg*100:.2f}%')


random Recall@50 baseline ~ 0.51%
VAL   Recall@50 16.94%   NDCG@50 5.36%
TEST  Recall@50 15.46%   NDCG@50 5.30%


In [46]:
# ---- save model + mappings + config (state_dict stores WEIGHTS, not architecture) ----
checkpoint = {
    'model_state': model.state_dict(),
    'user2idx': user2idx,
    'movie2idx': movie2idx,
    'config': {                                   # everything needed to rebuild the same architecture
        'n_emb': n_emb, 'n_head': n_head, 'n_layer': n_layer,
        'dropout': dropout, 'max_len': max_len,
        'user_size': len(user2idx), 'movie_size': len(movie2idx),
    },
}
torch.save(checkpoint, 'sasrec_model.pt')
print('saved sasrec_model.pt')


saved sasrec_model.pt


In [47]:
# ---- load (needs the class definitions Head/MultiHeadAttention/FeedForward/Block/SASRecModel) ----
checkpoint = torch.load('sasrec_model.pt', map_location=device)
user2idx  = checkpoint['user2idx']
movie2idx = checkpoint['movie2idx']

cfg = checkpoint['config']
# the classes read these GLOBALS when building -> restore them BEFORE creating the model
n_emb, n_head, n_layer = cfg['n_emb'], cfg['n_head'], cfg['n_layer']
dropout, max_len       = cfg['dropout'], cfg['max_len']

model = SASRecModel(cfg['user_size'], cfg['movie_size'])   # 1) build the right-shaped object
model.load_state_dict(checkpoint['model_state'])           # 2) fill weights by name
model = model.to(device)
model.eval()                                               # dropout OFF for inference
print('loaded sasrec_model.pt')


loaded sasrec_model.pt
